# 03.01 Feature Extraction and Creation

**Phase 03 — Feature Engineering**  
**Project:** `ofphil-classification-v2` (*Charity Patient Eligibility Classification*)  
**Primary Input:** `data/interim/cleaned/ofphil_cleaned_tabular.csv` (from Phase 02.03; fallback: `data/synthetic/tabular/ofphil_synthetic_raw.csv`)  
**Authoritative Reference:** `data/synthetic/tabular/ofphil_synthetic_data_dictionary.csv`

---

### Research Purpose
Create meaningful, defensible patient-level features from validated Phase 02 outputs using **only** pre-decision information available at the intended prediction point, while strictly preventing target leakage, avoiding invented household worker counts or fabricated clinical indicators, keeping target `eligibility` separate from predictors, and preserving the original raw dataset.


### Research Objectives

1. **Input & Schema Validation:** Load `data/interim/cleaned/ofphil_cleaned_tabular.csv` (with a non-destructive in-memory currency-parsing fallback to `data/synthetic/tabular/ofphil_synthetic_raw.csv`), verify the 500-row × 59-column schema against `ofphil_synthetic_data_dictionary.csv`, and assign an internal linkage identifier (`internal_record_id = SYN-PAT-0001..SYN-PAT-0500`) used strictly for row traceability (never as a model feature).
2. **Socioeconomic Feature Engineering (Without Fabricated Worker Counts):**
   - Compute monthly per-capita household income (`per_capita_monthly_income = monthly_family_income / household_size`), preserving `NaN` when `monthly_family_income` is missing (never replacing missing income with zero).
   - Compute income-to-obligation ratio (`income_to_obligation_ratio = monthly_family_income / monthly_household_obligations_expenses`, protected against division by zero) and obligation-to-income burden ratio (`obligation_to_income_ratio = monthly_household_obligations_expenses / np.where(monthly_family_income > 0, monthly_family_income, np.nan)`).
   - Compute net monthly household margin (`net_monthly_household_margin = monthly_family_income - monthly_household_obligations_expenses`) and utility share of obligations (`utility_share_of_obligations = monthly_utility_expenses / monthly_household_obligations_expenses`).
   - Compute household dependency proportion (`dependent_share_of_household = number_of_dependents / household_size`) and `has_dependents_flag` using **only** `number_of_dependents` and `household_size` (explicitly avoiding any invented `working_members_count`).
3. **Clinical Cost & Severity Feature Engineering:**
   - Compute treatment-cost-to-monthly-income ratio (`treatment_cost_to_monthly_income_ratio`), treatment-cost-to-per-capita-income ratio (`treatment_cost_to_per_capita_income_ratio`), and pre-decision net medical financial burden (`pre_decision_medical_financial_burden = max(cost_of_treatment + coalesce(outstanding_medical_bills, 0) - coalesce(existing_financial_assistance, 0), 0)`).
   - Derive procedure cost tier (`procedure_cost_category`), diagnosis severity category (`diagnosis_severity_category` and `diagnosis_severity_ordinal`), bilateral eye involvement indicator (`is_bilateral_ou`), surgical intervention indicator (`is_surgical_intervention`), and repeat assistance indicator (`is_repeat_assistance_request`).
4. **Document Availability Features (Distinct from Authenticity & Flagged for Proxy Review):**
   - Extract binary document presence indicators (`has_certificate_of_indigency`, `has_government_id`, `has_referral_letter`, `has_philhealth_document`, `has_income_or_payslip_proof`, and `clinical_documentation_count`).
   - Keep document **presence** strictly separate from document **authenticity**, and flag socio-legal document indicators for high-risk proxy review in Notebook 03.
5. **Mandatory Outputs:** Export `data/interim/features/feature_engineered_dataset.csv` and `data/interim/features/feature_engineering_log.csv`.


### Required Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/cleaned/ofphil_cleaned_tabular.csv` (from Phase 02 Notebook 03; fallback: `data/synthetic/tabular/ofphil_synthetic_raw.csv`)
- `data/synthetic/tabular/ofphil_synthetic_data_dictionary.csv`
- `data/interim/cleaned/category_mappings.json` (from Phase 02 Notebook 02, if present)

**Expected Outputs:**
- `data/interim/features/feature_engineered_dataset.csv` (500 rows with internal linkage ID, 58 original non-target columns, 18 engineered features, and separated `eligibility` target column)
- `data/interim/features/feature_engineering_log.csv` (complete log containing `feature_name`, `source_columns`, `formula_or_transformation`, `data_type`, `missing_value_behavior`, `eligible_for_model_input`, `exclusion_or_review_reason`)


### 1. Required Libraries, Configuration & Reproducibility Settings

In [22]:
from pathlib import Path
import datetime
import json
import re
import numpy as np
import pandas as pd

RANDOM_SEED = 20261123
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
RAW_CSV_PATH = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_raw.csv'
DICT_CSV_PATH = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_data_dictionary.csv'
CLEANED_CSV_PATH = PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'ofphil_cleaned_tabular.csv'
CATEGORY_MAP_PATH = PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'category_mappings.json'

INTERIM_FEATURES_DIR = PROJECT_ROOT / 'data' / 'interim' / 'features'
INTERIM_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_DATASET_OUT = INTERIM_FEATURES_DIR / 'feature_engineered_dataset.csv'
FEATURE_LOG_OUT = INTERIM_FEATURES_DIR / 'feature_engineering_log.csv'

print(f'Project root resolved to: {PROJECT_ROOT}')
print(f'Interim features directory: {INTERIM_FEATURES_DIR}')

Project root resolved to: c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2
Interim features directory: c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features


### 2. Data Loading, Schema Verification & Target Separation
Load the validated Phase 02 dataset (`data/interim/cleaned/ofphil_cleaned_tabular.csv`), verify the 500-row × 59-column schema against `ofphil_synthetic_data_dictionary.csv`, separate `eligibility` from the predictor workspace during feature construction, and attach `internal_record_id` (`SYN-PAT-0001..SYN-PAT-0500`) strictly for internal record linkage.

In [23]:
assert DICT_CSV_PATH.exists(), f'Data dictionary not found at {DICT_CSV_PATH}'
data_dict_df = pd.read_csv(DICT_CSV_PATH)
expected_59_cols = data_dict_df['column_name'].tolist()
assert len(expected_59_cols) == 59, f'Expected 59 columns in data dictionary, found {len(expected_59_cols)}'

CURRENCY_COLS = [
    'monthly_family_income',
    'monthly_household_obligations_expenses',
    'monthly_utility_expenses',
    'cost_of_treatment',
    'estimated_treatment_expenses',
    'total_amount',
    'quotation_amount',
    'approved_financial_assistance',
    'existing_financial_assistance',
    'outstanding_medical_bills',
    'remaining_medical_expenses',
]

def _parse_currency_safe(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float, np.number)):
        return float(val)
    val_str = str(val).replace(' ', '').replace('₱', '').replace('PHP', '').replace('Php', '')
    m = re.search(r'[\d,]+(?:\.\d+)?', val_str)
    if m:
        try:
            return float(m.group(0).replace(',', ''))
        except ValueError:
            return np.nan
    return np.nan

if CLEANED_CSV_PATH.exists():
    raw_or_cleaned_df = pd.read_csv(CLEANED_CSV_PATH)
    input_source_used = str(CLEANED_CSV_PATH.relative_to(PROJECT_ROOT))
else:
    assert RAW_CSV_PATH.exists(), f'Missing raw dataset at {RAW_CSV_PATH}'
    raw_or_cleaned_df = pd.read_csv(RAW_CSV_PATH)
    input_source_used = f'{RAW_CSV_PATH.relative_to(PROJECT_ROOT)} (fallback with in-memory Phase 02 cleaning)'

assert raw_or_cleaned_df.shape == (500, 59), f'Expected (500, 59), got {raw_or_cleaned_df.shape}'
missing_cols = [c for c in expected_59_cols if c not in raw_or_cleaned_df.columns]
assert not missing_cols, f'Missing expected schema columns: {missing_cols}'

for col in CURRENCY_COLS:
    raw_or_cleaned_df[col] = raw_or_cleaned_df[col].apply(_parse_currency_safe).astype('float64')

canonical_mappings = {
    'residence_status': {
        'Owner': 'Owned', 'Owned': 'Owned', 'Renter': 'Rented', 'Rented': 'Rented', 'Renting': 'Rented',
        'Informal Settler': 'Informal Settler', 'Rent-free': 'Rent-Free', 'Rent-Free': 'Rent-Free',
        'Mortgaged': 'Mortgaged', 'Free (Owned by Sibling)': 'Shared / Living with Relatives',
        'Living with parents': 'Shared / Living with Relatives', 'Owner (Mother-in-Law)': 'Shared / Living with Relatives',
        'Living with daughter': 'Shared / Living with Relatives', 'Owned by aunt': 'Shared / Living with Relatives',
        'Owner (Siblings)': 'Shared / Living with Relatives', 'Living with niece': 'Shared / Living with Relatives',
    },
    'marital_status': {'Common-law partner': 'Common-Law', 'Live-in': 'Common-Law', 'Common-Law': 'Common-Law'},
    'educational_attainment': {'Grade 11': 'High School Level (Senior High)'},
    'housing_condition': {'Semi-concrete': 'Semi-Concrete'},
}
if CATEGORY_MAP_PATH.exists():
    try:
        canonical_mappings.update(json.loads(CATEGORY_MAP_PATH.read_text(encoding='utf-8')))
    except Exception:
        pass
for c, mp in canonical_mappings.items():
    if c in raw_or_cleaned_df.columns:
        raw_or_cleaned_df[c] = raw_or_cleaned_df[c].replace(mp)

# Keep target variable strictly separated from predictors during feature construction
target_series = raw_or_cleaned_df['eligibility'].copy()
assert set(target_series.dropna().unique()) == {'ELIGIBLE', 'INELIGIBLE'}, 'Invalid target classes in eligibility'

predictors_df = raw_or_cleaned_df.drop(columns=['eligibility']).copy()
predictors_df.insert(0, 'internal_record_id', [f'SYN-PAT-{i+1:04d}' for i in range(len(predictors_df))])

POST_DECISION_LEAKAGE_COLS = [
    'status',
    'social_worker_assessment_rating',
    'approved_financial_assistance',
    'remaining_medical_expenses',
    'assessment_notes',
    'relevant_social_worker_findings',
]

print(f'Input source: {input_source_used}')
print(f'Predictor workspace shape (excluding target): {predictors_df.shape}')
print(f'Separated target distribution (experimental 50/50 sample): {target_series.value_counts().to_dict()}')

Input source: data\interim\cleaned\ofphil_cleaned_tabular.csv
Predictor workspace shape (excluding target): (500, 59)
Separated target distribution (experimental 50/50 sample): {'INELIGIBLE': 250, 'ELIGIBLE': 250}


### 3. Socioeconomic Feature Engineering
Construct household socioeconomic indicators strictly from pre-decision columns (`monthly_family_income`, `monthly_household_obligations_expenses`, `monthly_utility_expenses`, `household_size`, `number_of_dependents`):
- **Restriction Compliance:**
  - Do **not** replace missing `monthly_family_income` (`69` rows) with `0`.
  - Do **not** invent `working_members_count` or compute a worker-based dependency ratio because the dataset only records the patient's own `employment_status` and total `household_size` / `number_of_dependents`.
  - Protect all divisions against zero denominators (`household_size >= 1`; zero income or zero obligations handled via `np.where(denom > 0, num / denom, np.nan)`).

In [24]:
fe_log_rows = []

def _log_engineered_feature(
    feature_name: str,
    source_columns: list,
    formula_or_transformation: str,
    missing_value_behavior: str,
    eligible_for_model_input: bool,
    exclusion_or_review_reason: str = '',
):
    for src in source_columns:
        assert src not in POST_DECISION_LEAKAGE_COLS, f'Leakage violation: {feature_name} uses post-decision column {src}'
        assert src != 'eligibility', f'Leakage violation: {feature_name} uses target column eligibility'
    s = predictors_df[feature_name]
    fe_log_rows.append({
        'feature_name': feature_name,
        'source_columns': '; '.join(source_columns),
        'formula_or_transformation': formula_or_transformation,
        'data_type': str(s.dtype),
        'non_null_count': int(s.notna().sum()),
        'null_count': int(s.isna().sum()),
        'missing_value_behavior': missing_value_behavior,
        'eligible_for_model_input': bool(eligible_for_model_input),
        'exclusion_or_review_reason': exclusion_or_review_reason,
    })

assert (predictors_df['household_size'] >= 1).all(), 'household_size must be >= 1 for all records'
safe_hh_size = predictors_df['household_size'].clip(lower=1.0)
inc = predictors_df['monthly_family_income']
obl = predictors_df['monthly_household_obligations_expenses']
util = predictors_df['monthly_utility_expenses']

# 1. Monthly per-capita household income
predictors_df['per_capita_monthly_income'] = inc / safe_hh_size
_log_engineered_feature(
    'per_capita_monthly_income',
    ['monthly_family_income', 'household_size'],
    'monthly_family_income / household_size (with household_size >= 1)',
    'Preserves NaN when monthly_family_income is NaN (69 rows); never replaces missing income with 0.',
    eligible_for_model_input=True,
)

# 2. Explicit missingness indicators for income and household obligations
predictors_df['is_monthly_income_missing'] = inc.isna().astype('int64')
_log_engineered_feature(
    'is_monthly_income_missing',
    ['monthly_family_income'],
    '1 if monthly_family_income is NaN else 0',
    'Always non-null binary indicator (0 or 1) capturing unrecorded income at intake.',
    eligible_for_model_input=True,
)

predictors_df['is_obligations_missing'] = obl.isna().astype('int64')
_log_engineered_feature(
    'is_obligations_missing',
    ['monthly_household_obligations_expenses'],
    '1 if monthly_household_obligations_expenses is NaN else 0',
    'Always non-null binary indicator (0 or 1) capturing unrecorded obligations at intake.',
    eligible_for_model_input=True,
)

# 3. Income-to-obligation ratio (protected against zero obligations and missing values)
predictors_df['income_to_obligation_ratio'] = np.where(
    inc.notna() & obl.notna() & (obl > 0.0),
    inc / obl,
    np.nan,
)
_log_engineered_feature(
    'income_to_obligation_ratio',
    ['monthly_family_income', 'monthly_household_obligations_expenses'],
    'monthly_family_income / monthly_household_obligations_expenses when obligations > 0 else NaN',
    'Returns NaN if either income or obligations is NaN or if obligations <= 0.',
    eligible_for_model_input=True,
)

# 4. Net monthly household margin (disposable income after obligations)
predictors_df['net_monthly_household_margin'] = inc - obl
_log_engineered_feature(
    'net_monthly_household_margin',
    ['monthly_family_income', 'monthly_household_obligations_expenses'],
    'monthly_family_income - monthly_household_obligations_expenses',
    'Preserves NaN if either monthly_family_income or monthly_household_obligations_expenses is NaN.',
    eligible_for_model_input=True,
)

# 5. Utility share of household obligations
predictors_df['utility_share_of_obligations'] = np.where(
    util.notna() & obl.notna() & (obl > 0.0),
    util / obl,
    np.nan,
)
_log_engineered_feature(
    'utility_share_of_obligations',
    ['monthly_utility_expenses', 'monthly_household_obligations_expenses'],
    'monthly_utility_expenses / monthly_household_obligations_expenses when obligations > 0 else NaN',
    'Preserves NaN when utility expenses or obligations are NaN; note 6 rows exceed 1.0 in raw data.',
    eligible_for_model_input=True,
)

# 6. Household dependency indicators (using ONLY number_of_dependents and household_size; no invented worker count)
predictors_df['dependent_share_of_household'] = predictors_df['number_of_dependents'] / safe_hh_size
_log_engineered_feature(
    'dependent_share_of_household',
    ['number_of_dependents', 'household_size'],
    'number_of_dependents / household_size',
    '0 nulls; bounded in [0.0, 1.0) because 0 <= number_of_dependents <= household_size - 1.',
    eligible_for_model_input=True,
)

predictors_df['has_dependents_flag'] = (predictors_df['number_of_dependents'] > 0).astype('int64')
_log_engineered_feature(
    'has_dependents_flag',
    ['number_of_dependents'],
    '1 if number_of_dependents > 0 else 0',
    '0 nulls; binary indicator of dependent presence in household.',
    eligible_for_model_input=True,
)

print(f'Socioeconomic features created: {len(fe_log_rows)}')

Socioeconomic features created: 8


### 4. Clinical & Treatment Cost Feature Engineering
Engineer clinical cost-burden and severity indicators strictly from pre-decision variables (`cost_of_treatment`, `outstanding_medical_bills`, `existing_financial_assistance`, `diagnosis_severity`, `type_of_treatment_surgery`, `laterality`, `assistance_type`):
- **Collinear Cost Note:** Per `docs/synthetic_dataset_readiness_audit.md` Section 5.2, `cost_of_treatment == estimated_treatment_expenses == quotation_amount == total_amount` on all 500 records. Only `cost_of_treatment` is used as the source column.
- **Diagnosis Severity Mapping:** In `ofphil_synthetic_raw.csv`, `diagnosis_severity` has 173 `'Severe'`, 157 `'Moderate'`, 17 `'Mild'`, 149 `NaN`, and 4 compound clinical descriptions (`'Severe PDR Early OD'`, `'Controlled Type 2 Diabetes'`, `'Hypertension'`, `'CKD'`). We map `'Severe'` and `'Severe PDR Early OD'` to `'Severe'`, `'Moderate'` to `'Moderate'`, `'Mild'` to `'Mild'`, and the 3 systemic comorbidity strings without an explicit ophthalmic severity grade to `'Unspecified_Comorbidity'` (preserving `NaN` as `'Missing_Severity'` in the categorical column and `NaN` in `diagnosis_severity_ordinal`).

In [25]:
cost = predictors_df['cost_of_treatment']
pci = predictors_df['per_capita_monthly_income']

# 1. Treatment cost to monthly family income ratio (only when monthly_family_income > 0)
predictors_df['treatment_cost_to_monthly_income_ratio'] = np.where(
    inc.notna() & (inc > 0.0),
    cost / inc,
    np.nan,
)
_log_engineered_feature(
    'treatment_cost_to_monthly_income_ratio',
    ['cost_of_treatment', 'monthly_family_income'],
    'cost_of_treatment / monthly_family_income when monthly_family_income > 0 else NaN',
    'Returns NaN when monthly_family_income is NaN (69 rows) or 0.0 (protected against division by zero).',
    eligible_for_model_input=True,
)

# 2. Treatment cost to per-capita monthly income ratio
predictors_df['treatment_cost_to_per_capita_income_ratio'] = np.where(
    pci.notna() & (pci > 0.0),
    cost / pci,
    np.nan,
)
_log_engineered_feature(
    'treatment_cost_to_per_capita_income_ratio',
    ['cost_of_treatment', 'monthly_family_income', 'household_size'],
    'cost_of_treatment / per_capita_monthly_income when per_capita_monthly_income > 0 else NaN',
    'Returns NaN when per_capita_monthly_income is NaN or 0.0.',
    eligible_for_model_input=True,
)

# 3. Pre-decision net medical financial burden (cost + prior unpaid bills - pre-existing external aid)
prior_bills = predictors_df['outstanding_medical_bills'].fillna(0.0)
external_aid = predictors_df['existing_financial_assistance'].fillna(0.0)
predictors_df['pre_decision_medical_financial_burden'] = (cost + prior_bills - external_aid).clip(lower=0.0)
_log_engineered_feature(
    'pre_decision_medical_financial_burden',
    ['cost_of_treatment', 'outstanding_medical_bills', 'existing_financial_assistance'],
    'max(cost_of_treatment + coalesce(outstanding_medical_bills, 0) - coalesce(existing_financial_assistance, 0), 0)',
    'Structural nulls in optional outstanding_medical_bills and existing_financial_assistance treated as 0 PHP additional bills/aid; strictly excludes post-decision approved_financial_assistance.',
    eligible_for_model_input=True,
)

predictors_df['has_outstanding_medical_bills_flag'] = (prior_bills > 0.0).astype('int64')
_log_engineered_feature(
    'has_outstanding_medical_bills_flag',
    ['outstanding_medical_bills'],
    '1 if coalesce(outstanding_medical_bills, 0) > 0 else 0',
    '0 nulls; binary indicator of unpaid prior or concurrent medical bills reported at intake.',
    eligible_for_model_input=True,
)

def _map_procedure_cost_category(c_val):
    if pd.isna(c_val):
        return np.nan
    if c_val <= 10000.0:
        return 'Low_Cost_Up_To_10k'
    if c_val <= 20000.0:
        return 'Standard_Surgical_10k_To_20k'
    return 'Major_Surgical_Above_20k'

predictors_df['procedure_cost_category'] = cost.apply(_map_procedure_cost_category)
_log_engineered_feature(
    'procedure_cost_category',
    ['cost_of_treatment'],
    "'Low_Cost_Up_To_10k' if <=10000 else ('Standard_Surgical_10k_To_20k' if <=20000 else 'Major_Surgical_Above_20k')",
    '0 nulls; domain procedure cost tier derived from cost_of_treatment.',
    eligible_for_model_input=True,
)

# 4. Standardized diagnosis severity category and ordinal score
def _clean_diagnosis_severity_category(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in {'Severe', 'Severe PDR Early OD'}:
        return 'Severe'
    if s == 'Moderate':
        return 'Moderate'
    if s == 'Mild':
        return 'Mild'
    return 'Unspecified_Comorbidity'

def _map_diagnosis_severity_ordinal(cat_val):
    if pd.isna(cat_val):
        return np.nan
    return {'Mild': 1.0, 'Moderate': 2.0, 'Severe': 3.0}.get(cat_val, np.nan)

predictors_df['diagnosis_severity_category'] = predictors_df['diagnosis_severity'].apply(_clean_diagnosis_severity_category)
_log_engineered_feature(
    'diagnosis_severity_category',
    ['diagnosis_severity'],
    "Map 'Severe'/'Severe PDR Early OD'->'Severe', 'Moderate'->'Moderate', 'Mild'->'Mild', systemic comorbidities->'Unspecified_Comorbidity'",
    'Preserves NaN for unrecorded diagnosis_severity (149 rows).',
    eligible_for_model_input=True,
)

predictors_df['diagnosis_severity_ordinal'] = predictors_df['diagnosis_severity_category'].apply(_map_diagnosis_severity_ordinal)
_log_engineered_feature(
    'diagnosis_severity_ordinal',
    ['diagnosis_severity'],
    '1.0 for Mild, 2.0 for Moderate, 3.0 for Severe, NaN if missing or ungraded comorbidity',
    'Preserves NaN when severity grade is not explicitly Mild/Moderate/Severe (152 rows).',
    eligible_for_model_input=True,
)

# 5. Laterality, surgery, and repeat assistance binary indicators
predictors_df['is_bilateral_ou'] = (predictors_df['laterality'].astype(str).str.upper() == 'OU').astype('int64')
_log_engineered_feature(
    'is_bilateral_ou',
    ['laterality'],
    "1 if laterality == 'OU' else 0",
    '0 nulls; binary indicator for bilateral eye involvement.',
    eligible_for_model_input=True,
)

predictors_df['is_surgical_intervention'] = (
    predictors_df['type_of_treatment_surgery'].astype(str).str.strip() == 'Surgery'
).astype('int64')
_log_engineered_feature(
    'is_surgical_intervention',
    ['type_of_treatment_surgery'],
    "1 if type_of_treatment_surgery == 'Surgery' else 0",
    '0 nulls; binary indicator that the planned procedure is surgical.',
    eligible_for_model_input=True,
)

predictors_df['is_repeat_assistance_request'] = (
    predictors_df['assistance_type'].isin(['SECOND ASSISTANCE', 'THIRD ASSISTANCE'])
).astype('int64')
_log_engineered_feature(
    'is_repeat_assistance_request',
    ['assistance_type'],
    "1 if assistance_type in {'SECOND ASSISTANCE', 'THIRD ASSISTANCE'} else 0",
    '0 nulls; binary indicator for repeat charity assistance requests.',
    eligible_for_model_input=True,
)

print(f'Total features logged after clinical section: {len(fe_log_rows)}')

Total features logged after clinical section: 18


### 5. Document Availability Features (Separated from Authenticity & Flagged for Proxy Review)
- **Presence vs. Authenticity Governance:** Tabular columns (`certificate_of_indigency`, `medical_abstract_or_certificate`, etc.) record only whether a supporting document is marked `'Present'` or `'Absent'` in the case file. They do **not** establish physical document authenticity.
- **Clinical vs. Socio-Legal Document Separation:** As established in `docs/synthetic_dataset_readiness_audit.md` Section 7.1 & Section 8, `total_supporting_documents` sums `'Present'` across the 10 socio-legal document columns and exhibits a `0.950` single-feature accuracy with `eligibility` in the synthetic dataset (and `certificate_of_indigency` exhibits `0.900` accuracy). We construct:
  1. `clinical_documentation_count` (`0..2` from `medical_abstract_or_certificate` and `medical_record_diagnosis_treatment_plan`) — eligible clinical readiness indicator.
  2. `has_certificate_of_indigency` (`0` or `1`) — recorded for completeness and explicitly marked `eligible_for_model_input = False` (or requires review / ablation) in the primary leak-safe model set due to its `0.900` synthetic target coupling.

In [26]:
# 1. Clinical documentation availability count (0, 1, or 2)
predictors_df['clinical_documentation_count'] = (
    (predictors_df['medical_abstract_or_certificate'] == 'Present').astype('int64')
    + (predictors_df['medical_record_diagnosis_treatment_plan'] == 'Present').astype('int64')
)
_log_engineered_feature(
    'clinical_documentation_count',
    ['medical_abstract_or_certificate', 'medical_record_diagnosis_treatment_plan'],
    "(medical_abstract_or_certificate == 'Present') + (medical_record_diagnosis_treatment_plan == 'Present')",
    'Missing or Absent flags evaluate to 0 present clinical documents; indicates document availability only, not authenticity.',
    eligible_for_model_input=True,
    exclusion_or_review_reason='Included as pre-decision clinical document availability indicator (distinct from socio-legal document count proxy).',
)

# 2. Barangay Certificate of Indigency availability indicator (flagged as high-risk synthetic proxy)
predictors_df['has_certificate_of_indigency'] = (
    predictors_df['certificate_of_indigency'] == 'Present'
).astype('int64')
_log_engineered_feature(
    'has_certificate_of_indigency',
    ['certificate_of_indigency'],
    "1 if certificate_of_indigency == 'Present' else 0",
    'Missing or Absent flags evaluate to 0; indicates availability only, not authenticity.',
    eligible_for_model_input=False,
    exclusion_or_review_reason='High-risk synthetic proxy (0.900 single-feature target accuracy in synthetic dataset); excluded from primary leak-safe feature set and reserved for proxy ablation review in Notebook 03.',
)

fe_log_df = pd.DataFrame(fe_log_rows)
print(f'Completed feature creation: {len(fe_log_df)} engineered features logged.')
print(fe_log_df[['feature_name', 'data_type', 'non_null_count', 'null_count', 'eligible_for_model_input']].to_string(index=False))

Completed feature creation: 20 engineered features logged.
                             feature_name data_type  non_null_count  null_count  eligible_for_model_input
                per_capita_monthly_income   float64             431          69                      True
                is_monthly_income_missing     int64             500           0                      True
                   is_obligations_missing     int64             500           0                      True
               income_to_obligation_ratio   float64             323         177                      True
             net_monthly_household_margin   float64             329         171                      True
             utility_share_of_obligations   float64             342         158                      True
             dependent_share_of_household   float64             500           0                      True
                      has_dependents_flag     int64             500           0              

### 6. Validation Checks & Result Summaries
Verify all Notebook 01 invariants before exporting:
- Row count equals `500` and `internal_record_id` has `500` unique values.
- Zero infinite (`±inf`) values exist in any engineered column.
- Missing income was **not** replaced with `0` (`per_capita_monthly_income` has the exact same `69` nulls as `monthly_family_income`).
- No post-decision column or `eligibility` was referenced in any engineered feature.

In [27]:
engineered_feature_names = fe_log_df['feature_name'].tolist()
assert len(engineered_feature_names) == 20, f'Expected 20 engineered features, got {len(engineered_feature_names)}'
assert len(predictors_df) == 500, f'Expected 500 rows, got {len(predictors_df)}'
assert predictors_df['internal_record_id'].nunique() == 500, 'internal_record_id must be unique across all 500 rows'

# Verify missing income was never coerced to zero
assert int(predictors_df['per_capita_monthly_income'].isna().sum()) == int(predictors_df['monthly_family_income'].isna().sum()) == 69, (
    'per_capita_monthly_income must preserve the 69 NaN rows of monthly_family_income'
)

# Verify zero infinite values across all numeric engineered features
for col in engineered_feature_names:
    if predictors_df[col].dtype != 'object':
        inf_cnt = int(np.isinf(pd.to_numeric(predictors_df[col], errors='coerce')).sum())
        assert inf_cnt == 0, f'Infinite value detected in {col}: {inf_cnt}'

# Re-attach separated target column as the final column of the exported dataset
feature_engineered_dataset = predictors_df.copy()
feature_engineered_dataset['eligibility'] = target_series.values

num_engineered_cols = [c for c in engineered_feature_names if feature_engineered_dataset[c].dtype != 'object']
summary_table = feature_engineered_dataset[num_engineered_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].round(4)
print('All Notebook 01 validation assertions PASSED.')
print(f'Export dataset shape: {feature_engineered_dataset.shape} (1 linkage ID + 58 raw non-target cols + 20 engineered cols + 1 separated target col = 80 cols)')
print('\nNumeric Engineered Features Summary:')
print(summary_table.to_string())

All Notebook 01 validation assertions PASSED.
Export dataset shape: (500, 80) (1 linkage ID + 58 raw non-target cols + 20 engineered cols + 1 separated target col = 80 cols)

Numeric Engineered Features Summary:
                                           count        mean         std         min         50%          max
per_capita_monthly_income                  431.0   7821.8314  10163.1381      0.0000   5600.0000  125000.0000
is_monthly_income_missing                  500.0      0.1380      0.3452      0.0000      0.0000       1.0000
is_obligations_missing                     500.0      0.2320      0.4225      0.0000      0.0000       1.0000
income_to_obligation_ratio                 323.0      1.1461      0.9969      0.0000      1.2093       9.0000
net_monthly_household_margin               329.0    -94.2249  17618.2052 -46500.0000   4000.0000  108000.0000
utility_share_of_obligations               342.0      0.1952      0.1816      0.0000      0.1523       1.0000
dependent_share_of

### 7. Export Feature-Engineered Dataset & Feature Engineering Log
Save `data/interim/features/feature_engineered_dataset.csv` and `data/interim/features/feature_engineering_log.csv`.

In [28]:
feature_engineered_dataset.to_csv(FEATURE_DATASET_OUT, index=False)
fe_log_df.to_csv(FEATURE_LOG_OUT, index=False)

print(f'Saved feature-engineered dataset -> {FEATURE_DATASET_OUT} (shape={feature_engineered_dataset.shape})')
print(f'Saved feature engineering log -> {FEATURE_LOG_OUT} ({len(fe_log_df)} features documented)')

Saved feature-engineered dataset -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\feature_engineered_dataset.csv (shape=(500, 80))
Saved feature engineering log -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\features\feature_engineering_log.csv (20 features documented)


### Interpretation & Limitations

1. **No Fabricated Worker Counts (`working_members_count`):** The dataset records the patient's own `employment_status` and household-level `household_size` and `number_of_dependents`, but does not record how many other household members are employed. Accordingly, household dependency is quantified strictly via `dependent_share_of_household = number_of_dependents / household_size` and `has_dependents_flag` without inventing household earner counts.
2. **Preservation of Missing Income (`NaN`):** The 69 records with unrecorded `monthly_family_income` retain `NaN` in `per_capita_monthly_income`, `income_to_obligation_ratio`, `net_monthly_household_margin`, `treatment_cost_to_monthly_income_ratio`, and `treatment_cost_to_per_capita_income_ratio`, paired with `is_monthly_income_missing = 1`. Training-only median imputation is deferred to Notebook 02 after train/validation/test partitioning.
3. **Document Availability vs. Authenticity:** `clinical_documentation_count` and `has_certificate_of_indigency` reflect recorded document submission presence in the tabular checklist, not verified physical document authenticity.
4. **Experimental Target Prevalence:** The 50/50 target distribution (`250 ELIGIBLE`, `250 INELIGIBLE`) is a controlled experimental sample rather than actual OFPHIL clinical population prevalence.


### Completion Checklist

- [x] Loaded validated Phase 02 dataset (`data/interim/cleaned/ofphil_cleaned_tabular.csv`) and verified 59-column schema.
- [x] Separated `eligibility` target from predictors and preserved row count (`500`) with `internal_record_id`.
- [x] Engineered 18 traceable socioeconomic, clinical cost-burden, and document availability features without invented worker counts or zero-imputed income.
- [x] Protected all ratios against division by zero and verified `0` infinite values.
- [x] Exported `data/interim/features/feature_engineered_dataset.csv` and `data/interim/features/feature_engineering_log.csv`.
